## 1. 生存性数据图构建

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os, json, glob, re
from typing import Dict, List, Optional, Tuple, Set

import numpy as np
import pandas as pd
import torch
import dgl
import h5py

# =========================
# paths
# =========================
dataset = "LUSC"
WSI_ROOT = f"/data5/zhangye/morn/data/wsi/{dataset}"
OMICS_DIR = f"/data5/zhangye/morn/data/raw/{dataset}"

OMICS_FILES = {
    "CNV":   os.path.join(OMICS_DIR, f"{dataset}_CNV_top.csv"),
    "Methy": os.path.join(OMICS_DIR, f"{dataset}_Methy_top.csv"),
    "mRNA":  os.path.join(OMICS_DIR, f"{dataset}_mRNA_top.csv"),
    "miRNA": os.path.join(OMICS_DIR, f"{dataset}_miRNA_top.csv"),
}

GMT_PATH = "/data5/zhangye/morn/data/ReactomePathways.gmt"
MTI_PATH = "/data5/zhangye/morn/data/hsa_MTI.csv"

WSI_H5_ROOT = f"/data5/zhangye/trident/trident_processed_{dataset}/20x_256px_0px_overlap/features_uni_v1"
LABEL_CSV   = f"/data5/zhangye/morn/data/label/{dataset}_survival_labels.csv"
SPLIT_DIR   = f"/data5/zhangye/morn/data/splits_5folds/tcga_{dataset}"

OUT_DIR = f"../data/processed/{dataset}_hgt_dataset_"
os.makedirs(OUT_DIR, exist_ok=True)

# =========================
# hyperparams
# =========================
TOPK_PATIENT_OMICS = 50
WSI_TOPK_PATCH = 4096   # ✅ (1) 保存 WSI 特征时 K=4096

HUB_PCT = 80.0
MAX_HUBS_PER_PATHWAY = 5
SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

# CNV / Methy gene -> mRNA hub edge 的权重统计方式
CNV_USE_ABS_MEAN = True
METHY_USE_ABS_MEAN = True

# =========================
# helpers
# =========================
def list_patients_from_wsi_dir(wsi_root: str) -> List[str]:
    pts = []
    for name in os.listdir(wsi_root):
        p = os.path.join(wsi_root, name)
        if os.path.isdir(p) and name.startswith("TCGA-"):
            pts.append(name)
    return sorted(list(set(pts)))

def normalize_omics_pid(pid: str) -> Optional[str]:
    # TCGA-like: A.B.C... -> A-B-C
    parts = str(pid).split(".")
    if len(parts) < 3 or len(parts) > 4:
        return None
    return ".".join(parts[:3]).replace(".", "-")

def read_omics_csv(path: str) -> pd.DataFrame:
    assert os.path.isfile(path), f"Not found: {path}"
    return pd.read_csv(path, index_col=0)

def _norm_gene(x: str) -> str:
    return str(x).strip().upper() if x is not None else ""

def _norm_mirna(x: str, drop_arm: bool = True) -> str:
    x = str(x).strip().lower()
    if x == "":
        return ""
    x = x.replace("mirna", "mir")
    if x.startswith("mir-"):
        x = "hsa-" + x
    if drop_arm:
        x = re.sub(r"-(3p|5p)$", "", x)
    return x

def load_gmt(gmt_path: str) -> Dict[str, Set[str]]:
    assert os.path.isfile(gmt_path), f"Not found: {gmt_path}"
    pw = {}
    with open(gmt_path, "r") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 3:
                continue
            name = parts[0].strip()
            genes = set(_norm_gene(g) for g in parts[2:] if str(g).strip())
            genes = {g for g in genes if g}
            if name and genes:
                pw[name] = genes
    return pw

def _read_first_2d_dataset_from_h5(h5_path: str) -> np.ndarray:
    with h5py.File(h5_path, "r") as f:
        for key in ["features", "feat", "embeddings", "patch_features"]:
            if key in f and hasattr(f[key], "shape") and len(f[key].shape) == 2:
                return f[key][:]
        # fallback: first 2D dataset
        def _visit(name, obj):
            if hasattr(obj, "shape") and len(obj.shape) == 2:
                raise RuntimeError(name)
        try:
            f.visititems(_visit)
        except RuntimeError as e:
            k = str(e)
            return f[k][:]
    raise ValueError(f"No 2D dataset found in {h5_path}")

def _find_h5_for_patient(pid: str) -> Optional[str]:
    cand = sorted(glob.glob(os.path.join(WSI_H5_ROOT, f"{pid}-*.h5")))
    return cand[0] if len(cand) > 0 else None

def wsi_topk_keep_patches(h5_path: str, topk: int) -> Tuple[np.ndarray, np.ndarray]:
    feats = _read_first_2d_dataset_from_h5(h5_path).astype(np.float32)  # (N,D)
    n, d = feats.shape
    k = min(topk, n)
    out = np.zeros((topk, d), dtype=np.float32)
    mask = np.zeros((topk,), dtype=np.bool_)

    if n == 0 or k == 0:
        return out, mask

    scores = np.linalg.norm(feats, axis=1)
    idx = np.argpartition(scores, -k)[-k:]
    out[:k] = feats[idx]
    mask[:k] = True
    return out, mask

# -------------------------
# ✅ labels + survival loader (方案2：写进图里)
# -------------------------
def _pick_col(cols: List[str], candidates: List[str]) -> Optional[str]:
    cols_l = {c.lower(): c for c in cols}
    for cand in candidates:
        if cand.lower() in cols_l:
            return cols_l[cand.lower()]
    return None

def _infer_patient_col(df: pd.DataFrame) -> Optional[str]:
    candidates = [
        "patient_id", "case_id", "patient", "pid", "submitter_id",
        "bcr_patient_barcode", "bcr_patient_uuid", "sample", "barcode"
    ]
    return _pick_col(list(df.columns), candidates)

def _normalize_pid_like_tcga(x: str) -> str:
    s = str(x).strip()
    # allow TCGA-XX-XXXX-... -> TCGA-XX-XXXX
    if s.startswith("TCGA-"):
        parts = s.split("-")
        if len(parts) >= 3:
            return "-".join(parts[:3])
    # allow A.B.C -> A-B-C
    if "." in s:
        p = normalize_omics_pid(s)
        if p is not None:
            return p
    return s

def load_labels_and_survival(label_csv: str, patients: List[str]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Return:
      label_disc (long, -1 for missing)
      event_time (float)
      censorship (float)   # 1=censored, 0=event
    """
    assert os.path.isfile(label_csv), f"Not found: {label_csv}"
    df = pd.read_csv(label_csv)

    # set patient id index
    pid_col = _infer_patient_col(df)
    if pid_col is None:
        # fallback: try use the 2nd column as you used before (index_col=1)
        if df.shape[1] >= 2:
            pid_col = df.columns[1]
        else:
            raise ValueError(f"Cannot infer patient id column from {label_csv}. columns={list(df.columns)}")

    df = df.copy()
    df[pid_col] = df[pid_col].astype(str).map(_normalize_pid_like_tcga)
    df = df.set_index(pid_col, drop=True)

    # required label_disc
    label_col = _pick_col(list(df.columns), ["label_disc", "label", "y_bin", "y", "disc_label"])
    if label_col is None:
        raise KeyError(f"label file missing label column. tried [label_disc,label,y_bin,...], got={list(df.columns)}")

    # time column
    time_col = _pick_col(list(df.columns), ["event_time", "time", "survival_months", "OS_months", "OS", "months", "t"])
    if time_col is None:
        raise KeyError(f"label file missing time column. tried [event_time,time,OS_months,...], got={list(df.columns)}")

    # censorship preferred; else derive from event/status
    censor_col = _pick_col(list(df.columns), ["censorship", "censored", "censor"])
    event_col  = _pick_col(list(df.columns), ["event", "status", "dead", "death", "vital_status"])

    mp_label = df[label_col].to_dict()
    mp_time  = df[time_col].to_dict()

    if censor_col is not None:
        mp_censor = df[censor_col].to_dict()
        def _get_censor(pid: str) -> float:
            v = mp_censor.get(pid, np.nan)
            if pd.isna(v):
                return float("nan")
            return float(v)
    else:
        if event_col is None:
            raise KeyError(
                f"label file missing censorship column AND missing event/status column; "
                f"need one of them. columns={list(df.columns)}"
            )
        mp_event = df[event_col].to_dict()
        # assume event=1 means event happened (death), so censorship = 1 - event
        def _get_censor(pid: str) -> float:
            v = mp_event.get(pid, np.nan)
            if pd.isna(v):
                return float("nan")
            ev = float(v)
            # sometimes vital_status might be 'Alive'/'Dead'
            if isinstance(v, str):
                s = str(v).strip().lower()
                if s in ["dead", "deceased", "1", "true"]:
                    ev = 1.0
                elif s in ["alive", "0", "false"]:
                    ev = 0.0
            return float(1.0 - ev)

    y_list, t_list, c_list = [], [], []
    for p in patients:
        yv = mp_label.get(p, np.nan)
        tv = mp_time.get(p, np.nan)
        cv = _get_censor(p)

        y_list.append(int(yv) if (p in mp_label and pd.notna(yv)) else -1)
        t_list.append(float(tv) if (p in mp_time and pd.notna(tv)) else float("nan"))
        c_list.append(float(cv) if not (isinstance(cv, float) and np.isnan(cv)) else float("nan"))

    y = torch.tensor(y_list, dtype=torch.long)
    t = torch.tensor(t_list, dtype=torch.float32)
    c = torch.tensor(c_list, dtype=torch.float32)
    return y, t, c

def load_split_indices(split_csv: str, patients: List[str]) -> Dict[str, torch.Tensor]:
    df = pd.read_csv(split_csv)
    patient2idx = {p: i for i, p in enumerate(patients)}

    def _col(col: str) -> torch.Tensor:
        pids = df[col].dropna().astype(str).str.strip().tolist()
        pids = [_normalize_pid_like_tcga(x) for x in pids]
        kept = [patient2idx[p] for p in pids if p in patient2idx]
        return torch.tensor(kept, dtype=torch.int64)

    for c in ["train", "val", "test"]:
        assert c in df.columns, f"Split file missing column {c}: got {list(df.columns)}"
    return {"train_idx": _col("train"), "val_idx": _col("val"), "test_idx": _col("test")}

# =========================
# OMICS: build patient-gene edges + weights
# =========================
def build_patient_gene_edges(
    df_pat_x_feat_T: pd.DataFrame,
    patients_keep: List[str],
    topk: int,
    rank_use_abs: bool,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], np.ndarray]:
    """
    df_pat_x_feat_T: rows=patients(raw id), cols=features(gene/miRNA)
    Return:
      u(patient_idx), v(gene_idx), w(edge weight from value), col_names, X_aligned(P,G)
    """
    df = df_pat_x_feat_T.copy()
    df["__pid__"] = [normalize_omics_pid(x) for x in df.index]
    df = df[df["__pid__"].notna()].set_index("__pid__", drop=True)

    feats = df.drop(columns=[c for c in df.columns if str(c).startswith("__")], errors="ignore")
    col_names = list(feats.columns)

    P = len(patients_keep)
    G = len(col_names)

    # aligned matrix X(P,G) for later gene-level stats
    X = np.zeros((P, G), dtype=np.float32)
    pid2row = {pid: i for i, pid in enumerate(feats.index)}
    for i, pid in enumerate(patients_keep):
        if pid in pid2row:
            X[i] = feats.iloc[pid2row[pid]].to_numpy(dtype=np.float32)

    us, vs, ws = [], [], []
    for p_idx in range(P):
        row = X[p_idx]
        score = np.abs(row) if rank_use_abs else row
        k = min(topk, score.shape[0])
        if k <= 0:
            continue
        idx = np.argpartition(score, -k)[-k:]
        us.extend([p_idx] * len(idx))
        vs.extend(idx.astype(np.int64).tolist())
        ws.extend(row[idx].astype(np.float32).tolist())  # ✅ weight = 原始值（表达/数值）

    return np.array(us, np.int64), np.array(vs, np.int64), np.array(ws, np.float32), col_names, X

# =========================
# MTI: miRNA->mRNA edges + hub mRNA
# =========================
def load_mti_edges(
    mti_path: str,
    mir_cols: List[str],
    mrna_cols: List[str],
    drop_arm: bool = True,
) -> Tuple[np.ndarray, np.ndarray, Set[str]]:
    df = pd.read_csv(mti_path)
    assert "miRNA" in df.columns and "Target Gene" in df.columns, f"Bad MTI columns: {list(df.columns)}"

    print('mir col', mir_cols[:20])
    print('mrna col', mrna_cols[:20])
    print('=' * 100)

    mir_map = {}
    for i, m in enumerate(mir_cols):
        k = _norm_mirna(m, drop_arm=drop_arm)
        if k and k not in mir_map:
            mir_map[k] = i

    mrna_map = {}
    for i, g in enumerate(mrna_cols):
        k = _norm_gene(g)
        if k and k not in mrna_map:
            mrna_map[k] = i

    mir_s = df["miRNA"].astype(str).map(lambda x: _norm_mirna(x, drop_arm=drop_arm))
    gene_s = df["Target Gene"].astype(str).map(_norm_gene)

    U, V = [], []
    tgt_counts: Dict[str, int] = {}
    for m, g in zip(mir_s, gene_s):
        if g in mrna_map:
            tgt_counts[g] = tgt_counts.get(g, 0) + 1
        if (m in mir_map) and (g in mrna_map):
            U.append(mir_map[m])
            V.append(mrna_map[g])

    # hub mRNA by target count percentile
    hub_genes: Set[str] = set()
    if len(tgt_counts) > 0:
        vals = np.array(list(tgt_counts.values()), dtype=np.float32)
        thr = np.percentile(vals, HUB_PCT)
        hub_genes = {g for g, c in tgt_counts.items() if c >= thr}

    return np.array(U, np.int64), np.array(V, np.int64), hub_genes


def add_pathway_hub_edges(
    data_dict: dict,
    cnv_cols: List[str],
    met_cols: List[str],
    mrna_cols: List[str],
    pathway2genes: Dict[str, Set[str]],
    hub_mrna_genes: Set[str],
    max_hubs_per_pathway: int = 5,
):
    cnv_map   = {_norm_gene(g): i for i, g in enumerate(cnv_cols)}
    met_map   = {_norm_gene(g): i for i, g in enumerate(met_cols)}
    mrna_map  = {_norm_gene(g): i for i, g in enumerate(mrna_cols)}
    hub_norm  = {_norm_gene(g) for g in hub_mrna_genes}

    U_cnv, V_cnv = [], []
    U_met, V_met = [], []

    for _, genes in pathway2genes.items():
        hubs = [g for g in genes if (g in hub_norm) and (g in mrna_map)]
        if len(hubs) == 0:
            continue
        hubs = hubs[:max_hubs_per_pathway]
        hub_ids = [mrna_map[g] for g in hubs]

        for g in genes:
            if g in cnv_map:
                u = cnv_map[g]
                for v in hub_ids:
                    U_cnv.append(u); V_cnv.append(v)
            if g in met_map:
                u = met_map[g]
                for v in hub_ids:
                    U_met.append(u); V_met.append(v)

    if len(U_cnv) > 0:
        U = np.array(U_cnv, np.int64); V = np.array(V_cnv, np.int64)
        data_dict[("gene_CNV", "cnv_to_mrna", "gene_mRNA")] = (U, V)
        data_dict[("gene_mRNA", "mrna_to_cnv", "gene_CNV")] = (V, U)

    if len(U_met) > 0:
        U = np.array(U_met, np.int64); V = np.array(V_met, np.int64)
        data_dict[("gene_Methy", "methy_to_mrna", "gene_mRNA")] = (U, V)
        data_dict[("gene_mRNA", "mrna_to_methy", "gene_Methy")] = (V, U)

# =========================
# Main
# =========================
patients = list_patients_from_wsi_dir(WSI_ROOT)
print(f"[WSI] {dataset} patients = {len(patients)}")

P = len(patients)
num_nodes_dict = {"patient": P, "wsi": P}

data_dict = {}
edge_w = {}  # canonical etype -> weight array (float32)

# ---- patient <-> wsi (weight=1)
u = np.arange(P, dtype=np.int64)
v = np.arange(P, dtype=np.int64)
data_dict[("patient", "has_wsi", "wsi")] = (u, v)
data_dict[("wsi", "belongs_to", "patient")] = (v, u)
edge_w[("patient", "has_wsi", "wsi")] = np.ones((P,), np.float32)
edge_w[("wsi", "belongs_to", "patient")] = np.ones((P,), np.float32)

# ---- patient <-> omics
omics_cols = {}
omics_X = {}   # ntype -> X(P,G) aligned matrix

for om, fpath in OMICS_FILES.items():
    # 原文件通常是 gene x patient；你之前用 .T，所以这里保持一致：rows=patient, cols=gene
    df = read_omics_csv(fpath).T

    if om in ["CNV", "Methy"]:
        rank_use_abs = True
    else:
        rank_use_abs = False

    pu, gv, w_val, cols, X = build_patient_gene_edges(df, patients, TOPK_PATIENT_OMICS, rank_use_abs=rank_use_abs)

    if om == "CNV":
        ntype, e_fwd, e_rev = "gene_CNV", "has_cnv", "in_patient"
    elif om == "Methy":
        ntype, e_fwd, e_rev = "gene_Methy", "has_methy", "in_patient"
    elif om == "mRNA":
        ntype, e_fwd, e_rev = "gene_mRNA", "has_mrna", "in_patient"
    elif om == "miRNA":
        ntype, e_fwd, e_rev = "gene_miRNA", "has_mirna", "in_patient"
    else:
        raise ValueError(om)

    omics_cols[ntype] = cols
    omics_X[ntype] = X
    num_nodes_dict[ntype] = len(cols)

    data_dict[("patient", e_fwd, ntype)] = (pu, gv)
    data_dict[(ntype, e_rev, "patient")] = (gv, pu)

    # ✅ (2) 边权重规则：
    # - 只有 patient<->mRNA 用表达值；其余 patient<->omics 先设为 1
    if ntype == "gene_mRNA":
        edge_w[("patient", e_fwd, ntype)] = w_val.astype(np.float32)
        edge_w[(ntype, e_rev, "patient")] = w_val.astype(np.float32)
    else:
        edge_w[("patient", e_fwd, ntype)] = np.ones((len(pu),), np.float32)
        edge_w[(ntype, e_rev, "patient")] = np.ones((len(pu),), np.float32)

    print(f"[OMICS] {om}: nodes={len(cols)} | edges={len(pu)}")

# ---- miRNA -> mRNA edges (MTI), weight = miRNA expression (mean over patients)
print("[MTI] Building miRNA->mRNA edges...")
U_mti, V_mti, hub_mrna_genes = load_mti_edges(
    MTI_PATH,
    mir_cols=omics_cols["gene_miRNA"],
    mrna_cols=omics_cols["gene_mRNA"],
    drop_arm=True,
)
if len(U_mti) > 0:
    data_dict[("gene_miRNA", "targets", "gene_mRNA")] = (U_mti, V_mti)
    data_dict[("gene_mRNA", "targeted_by", "gene_miRNA")] = (V_mti, U_mti)

    # miRNA 表达值：对齐矩阵 X(P,Gmir)，取每个 miRNA 在所有病人上的均值
    X_mir = omics_X["gene_miRNA"]  # (P, Gmir)
    mir_mean = X_mir.mean(axis=0).astype(np.float32)  # (Gmir,)

    w_mti = mir_mean[U_mti].astype(np.float32)  # 每条边取 source miRNA 的表达值
    edge_w[("gene_miRNA", "targets", "gene_mRNA")] = w_mti
    edge_w[("gene_mRNA", "targeted_by", "gene_miRNA")] = w_mti
    print(f"[MTI] edges={len(U_mti)} (weight=miRNA mean expression)")
else:
    print("[WARN] MTI edges = 0 (name mismatch?)")

# ---- pathway hub edges: CNV/Methy -> hub mRNA
print("[GMT] Loading Reactome GMT + building CNV/Methy -> hub mRNA edges...")
pathway2genes = load_gmt(GMT_PATH)
add_pathway_hub_edges(
    data_dict,
    cnv_cols=omics_cols["gene_CNV"],
    met_cols=omics_cols["gene_Methy"],
    mrna_cols=omics_cols["gene_mRNA"],
    pathway2genes=pathway2genes,
    hub_mrna_genes=hub_mrna_genes,
    max_hubs_per_pathway=MAX_HUBS_PER_PATHWAY,
)

# ✅ Methy->mRNA 权重：Methy 表达（均值）
if ("gene_Methy", "methy_to_mrna", "gene_mRNA") in data_dict:
    U, V = data_dict[("gene_Methy", "methy_to_mrna", "gene_mRNA")]
    X_met = omics_X["gene_Methy"]
    met_mean = (np.abs(X_met).mean(axis=0) if METHY_USE_ABS_MEAN else X_met.mean(axis=0)).astype(np.float32)
    w = met_mean[U].astype(np.float32)
    edge_w[("gene_Methy", "methy_to_mrna", "gene_mRNA")] = w
    edge_w[("gene_mRNA", "mrna_to_methy", "gene_Methy")] = w
    print(f"[PATH-HUB] Methy->mRNA edges={len(U)} (weight=methy mean)")

# ✅ CNV->mRNA 权重：CNV 数值（均值；默认 abs mean）
if ("gene_CNV", "cnv_to_mrna", "gene_mRNA") in data_dict:
    U, V = data_dict[("gene_CNV", "cnv_to_mrna", "gene_mRNA")]
    X_cnv = omics_X["gene_CNV"]
    cnv_mean = (np.abs(X_cnv).mean(axis=0) if CNV_USE_ABS_MEAN else X_cnv.mean(axis=0)).astype(np.float32)
    w = cnv_mean[U].astype(np.float32)
    edge_w[("gene_CNV", "cnv_to_mrna", "gene_mRNA")] = w
    edge_w[("gene_mRNA", "mrna_to_cnv", "gene_CNV")] = w
    print(f"[PATH-HUB] CNV->mRNA edges={len(U)} (weight=cnv mean)")

# ---- build graph
g = dgl.heterograph(data_dict, num_nodes_dict=num_nodes_dict)
print("[GRAPH]", g)

# attach edge weights (other missing -> 1)
for et in g.canonical_etypes:
    n_e = g.num_edges(et)
    if et in edge_w:
        w = edge_w[et]
        assert len(w) == n_e, f"edge weight length mismatch for {et}: {len(w)} vs {n_e}"
        g.edges[et].data["w"] = torch.tensor(w, dtype=torch.float32)
    else:
        g.edges[et].data["w"] = torch.ones((n_e,), dtype=torch.float32)

# node ids
for ntype in g.ntypes:
    g.nodes[ntype].data["nid"] = torch.arange(g.num_nodes(ntype), dtype=torch.long)

# ---- WSI patch feats (K=4096)
print(f"[WSI-PATCH] Saving topK patches per WSI: K={WSI_TOPK_PATCH}")
missing_h5 = 0
feat_dim = None

wsi_patches_list = []
wsi_mask_list = []

for pid in patients:
    h5_path = _find_h5_for_patient(pid)
    if h5_path is None:
        missing_h5 += 1
        wsi_patches_list.append(None)
        wsi_mask_list.append(None)
        continue
    patches, mask = wsi_topk_keep_patches(h5_path, topk=WSI_TOPK_PATCH)
    feat_dim = feat_dim or int(patches.shape[1])
    wsi_patches_list.append(patches)
    wsi_mask_list.append(mask)

if feat_dim is None:
    raise RuntimeError("No WSI h5 features found. Check WSI_H5_ROOT.")

wsi_patches = np.zeros((P, WSI_TOPK_PATCH, feat_dim), dtype=np.float32)
wsi_mask    = np.zeros((P, WSI_TOPK_PATCH), dtype=np.bool_)
for i in range(P):
    if wsi_patches_list[i] is None:
        continue
    wsi_patches[i] = wsi_patches_list[i]
    wsi_mask[i]    = wsi_mask_list[i]

g.nodes["wsi"].data["wsi_patches"] = torch.tensor(wsi_patches, dtype=torch.float32)
g.nodes["wsi"].data["wsi_patch_mask"] = torch.tensor(wsi_mask, dtype=torch.bool)

if missing_h5 > 0:
    print(f"[WARN] missing h5 for {missing_h5}/{P} patients -> wsi_patches filled with zeros.")

# ---- labels + ✅ survival fields (方案2：写入图里)
labels, event_time, censorship = load_labels_and_survival(LABEL_CSV, patients)
g.nodes["patient"].data["label"] = labels
g.nodes["patient"].data["event_time"] = event_time
g.nodes["patient"].data["censorship"] = censorship
print("[LABEL] labeled patients =", int((labels >= 0).sum().item()))
print("[SURV] event_time/censorship saved into graph (patient node data)")

def _filter_labeled(idx: torch.Tensor) -> torch.Tensor:
    y = g.nodes["patient"].data["label"]
    return idx[y[idx] >= 0]

# ---- save graph
graph_path = os.path.join(OUT_DIR, f"{dataset}_graph.bin")
dgl.save_graphs(graph_path, [g])
print(f"[SAVED] {graph_path}")

# ---- save each fold split/meta
split_csvs = sorted(glob.glob(os.path.join(SPLIT_DIR, "splits_*.csv")))
if len(split_csvs) == 0:
    split_csvs = sorted(glob.glob(os.path.join(SPLIT_DIR, "*.csv")))
assert len(split_csvs) > 0, f"No split csv found under: {SPLIT_DIR}"
print(f"[SPLIT] found {len(split_csvs)} split files.")

for split_csv in split_csvs:
    fold_name = os.path.splitext(os.path.basename(split_csv))[0]
    fold_out = os.path.join(OUT_DIR, fold_name)
    os.makedirs(fold_out, exist_ok=True)

    splits = load_split_indices(split_csv, patients)
    train_idx = _filter_labeled(splits["train_idx"])
    val_idx   = _filter_labeled(splits["val_idx"])
    test_idx  = _filter_labeled(splits["test_idx"])

    torch.save({"train_idx": train_idx, "val_idx": val_idx, "test_idx": test_idx},
               os.path.join(fold_out, f"{fold_name}_split.pt"))

    meta = {
        "patients": patients,
        "num_nodes": {k: int(v) for k, v in num_nodes_dict.items()},
        "canonical_etypes": [list(x) for x in g.canonical_etypes],
        "topk_patient_omics": TOPK_PATIENT_OMICS,
        "wsi_topk_patch": WSI_TOPK_PATCH,
        "wsi_feat_dim": int(feat_dim),
        "label_csv": LABEL_CSV,
        "split_csv": split_csv,
        "graph_bin": graph_path,
        "edge_weight_rule": {
            "patient<->mRNA": "mRNA expression value per (patient,gene) edge",
            "miRNA->mRNA": "mean miRNA expression (over patients) on each edge",
            "Methy->mRNA": "mean Methy value (over patients) on each edge",
            "CNV->mRNA": "mean CNV value (over patients, abs_mean by default) on each edge",
            "others": "1",
        },
        "split_sizes_after_filter_labeled": {
            "train": int(len(train_idx)),
            "val": int(len(val_idx)),
            "test": int(len(test_idx)),
        }
    }
    with open(os.path.join(fold_out, f"{fold_name}_meta.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[FOLD {fold_name}] train/val/test = {len(train_idx)}/{len(val_idx)}/{len(test_idx)}")

print("[DONE]")


[WSI] LUSC patients = 356
[OMICS] CNV: nodes=5000 | edges=17800
[OMICS] Methy: nodes=5000 | edges=17800
[OMICS] mRNA: nodes=5000 | edges=17800
[OMICS] miRNA: nodes=200 | edges=17800
[MTI] Building miRNA->mRNA edges...
mir col ['hsa-mir-222', 'hsa-mir-3170', 'hsa-mir-653', 'hsa-mir-188', 'hsa-mir-143', 'hsa-mir-376a-1', 'hsa-mir-186', 'hsa-mir-758', 'hsa-mir-199a-1', 'hsa-mir-542', 'hsa-mir-659', 'hsa-mir-487a', 'hsa-mir-664a', 'hsa-mir-4677', 'hsa-mir-3653', 'hsa-mir-643', 'hsa-mir-3615', 'hsa-mir-1307', 'hsa-mir-889', 'hsa-mir-1269a']
mrna col ['MTHFD1L', 'AEBP2', 'MIR6717', 'FAM53C', 'LUC7L', 'RRAGD', 'MIR6743', 'SMU1', 'PKIA-AS1', 'SDSL', 'STX18', 'LOC283731', 'JPX', 'TSPAN3', 'LOC440173', 'ATXN2', 'CDKN1B', 'PLA2G10', 'FLJ46284', 'HEG1']
[MTI] edges=123336 (weight=miRNA mean expression)
[GMT] Loading Reactome GMT + building CNV/Methy -> hub mRNA edges...
[PATH-HUB] Methy->mRNA edges=124910 (weight=methy mean)
[PATH-HUB] CNV->mRNA edges=123547 (weight=cnv mean)
[GRAPH] Graph(num_nod

KeyboardInterrupt: 